### N-gram language models or how to write scientific papers (4 pts)

We shall train our language model on a corpora of [ArXiv](http://arxiv.org/) articles and see if we can generate a new one!

![img](https://media.npr.org/assets/img/2013/12/10/istock-18586699-monkey-computer_brick-16e5064d3378a14e0e4c2da08857efe03c04695e-s800-c85.jpg)

_data by neelshah18 from [here](https://www.kaggle.com/neelshah18/arxivdataset/)_

_Disclaimer: this has nothing to do with actual science. But it's fun, so who cares?!_

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# Alternative manual download link: https://yadi.sk/d/_nGyU2IajjR9-w
# !wget "https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1" -O arxivData.json.tar.gz
# !tar -xvzf arxivData.json.tar.gz
data = pd.read_json("./arxivData.json")
data.sample(n=5)

,author,day,id,link,month,summary,tag,title,year
11499,[{'name': 'Tor Lattimore'}],29,1603.08661v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",3,I introduce and analyse an anytime version of ...,"[{'term': 'cs.LG', 'scheme': 'http://arxiv.org...",Regret Analysis of the Anytime Optimally Confi...,2016
28137,"[{'name': 'Adriana Romero'}, {'name': 'Michal ...",21,1705.07450v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",5,Inspired by the combination of feedforward and...,"[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Image Segmentation by Iterative Inference from...,2017
12664,"[{'name': 'Yingzhen Yang'}, {'name': 'Xinqi Ch...",8,1210.4481v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",10,Image colorization adds color to grayscale ima...,"[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Epitome for Automatic Image Colorization,2012
6150,"[{'name': 'Ru-Ze Liang'}, {'name': 'Wei Xie'},...",16,1608.04581v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",8,"In this paper, we propose a novel learning fra...","[{'term': 'cs.LG', 'scheme': 'http://arxiv.org...",A novel transfer learning method based on comm...,2016
18106,[{'name': 'Jean-Marie Chauvet'}],16,cs/0402035v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",2,Recent advances in programming languages study...,"[{'term': 'cs.AI', 'scheme': 'http://arxiv.org...",Memory As A Monadic Control Construct In Probl...,2004


In [3]:
# assemble lines: concatenate title and description
lines = data.apply(lambda row: row['title'] + ' ; ' + row['summary'].replace("\n", ' '), axis=1).tolist()

sorted(lines, key=len)[:3]

['Differential Contrastive Divergence ; This paper has been retracted.',
 'What Does Artificial Life Tell Us About Death? ; Short philosophical essay',
 'P=NP ; We claim to resolve the P=?NP problem via a formal argument for P=NP.']

### Tokenization

You know the dril. The data is messy. Go clean the data. Use WordPunctTokenizer or something.


In [4]:
# Task: convert lines (in-place) into strings of space-separated tokens. Import & use WordPunctTokenizer
from nltk.tokenize import WordPunctTokenizer

tokenizer = WordPunctTokenizer()

lines = [' '.join(tokenizer.tokenize(line.lower())) for line in lines]

In [5]:
assert sorted(lines, key=len)[0] == \
    'differential contrastive divergence ; this paper has been retracted .'
assert sorted(lines, key=len)[2] == \
    'p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .'

### N-Gram Language Model (1point)

A language model is a probabilistic model that estimates text probability: the joint probability of all tokens $w_t$ in text $X$: $P(X) = P(w_1, \dots, w_T)$.

It can do so by following the chain rule:
$$P(w_1, \dots, w_T) = P(w_1)P(w_2 \mid w_1)\dots P(w_T \mid w_1, \dots, w_{T-1}).$$ 

The problem with such approach is that the final term $P(w_T \mid w_1, \dots, w_{T-1})$ depends on $n-1$ previous words. This probability is impractical to estimate for long texts, e.g. $T = 1000$.

One popular approximation is to assume that next word only depends on a finite amount of previous words:

$$P(w_t \mid w_1, \dots, w_{t - 1}) = P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1})$$

Such model is called __n-gram language model__ where n is a parameter. For example, in 3-gram language model, each word only depends on 2 previous words. 

$$
    P(w_1, \dots, w_n) = \prod_t P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1}).
$$

You can also sometimes see such approximation under the name of _n-th order markov assumption_.

The first stage to building such a model is counting all word occurences given N-1 previous words

In [6]:
from tqdm import tqdm
from collections import defaultdict, Counter

# special tokens: 
# - `UNK` represents absent tokens, 
# - `EOS` is a special token after the end of sequence

UNK, EOS = "_UNK_", "_EOS_"

def pad_prefix(prefix, n):
    d = n - 1 - len(prefix)  # |required_prefix| - |prefix|
    padded_prefix = [ UNK ] * max(0, d) + prefix[max(0, -d):]
    return tuple(padded_prefix)

def count_ngrams(lines, n):
    """
    Count how many times each word occured after (n - 1) previous words
    :param lines: an iterable of strings with space-separated tokens
    :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

    When building counts, please consider the following two edge cases:
    - if prefix is shorter than (n - 1) tokens, it should be padded with UNK. For n=3,
      empty prefix: "" -> (UNK, UNK)
      short prefix: "the" -> (UNK, the)
      long prefix: "the new approach" -> (new, approach)
    - you should add a special token, EOS, at the end of each sequence
      "... with deep neural networks ." -> (..., with, deep, neural, networks, ., EOS)
      count the probability of this token just like all others.
    """
    counts = defaultdict(Counter)
    # counts[(word1, word2)][word3] = how many times word3 occured after (word1, word2)

    for line in tqdm(lines):
        tokens = line.split() + [EOS]
        for i in range(len(tokens)):
            prefix, word = tokens[:i], tokens[i]
            padded_prefix = pad_prefix(prefix, n)
            counts[padded_prefix].update([word])
    
    return counts


In [7]:
# let's test it
dummy_lines = sorted(lines, key=len)[:100]
dummy_counts = count_ngrams(dummy_lines, n=3)
assert set(map(len, dummy_counts.keys())) == {2}, "please only count {n-1}-grams"
assert len(dummy_counts[('_UNK_', '_UNK_')]) == 78
assert dummy_counts['_UNK_', 'a']['note'] == 3
assert dummy_counts['p', '=']['np'] == 2
assert dummy_counts['author', '.']['_EOS_'] == 1

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 25475.61it/s]


Once we can count N-grams, we can build a probabilistic language model.
The simplest way to compute probabilities is in proporiton to counts:

$$ P(w_t | prefix) = { Count(prefix, w_t) \over \sum_{\hat w} Count(prefix, \hat w) } $$

In [8]:
class NGramLanguageModel:    
    def __init__(self, lines, n):
        """ 
        Train a simple count-based language model: 
        compute probabilities P(w_t | prefix) given ngram counts
        
        :param n: computes probability of next token given (n - 1) previous words
        :param lines: an iterable of strings with space-separated tokens
        """
        assert n >= 1
        self.n = n
    
        counts = count_ngrams(lines, self.n)
        
        # compute token proabilities given counts
        self.probs = defaultdict(Counter)
        # probs[(word1, word2)][word3] = P(word3 | word1, word2)
        
        # populate self.probs with actual probabilities
        for prefix in tqdm(counts):
            total = 0
            for word in counts[prefix]:
                total += counts[prefix][word]
            for word in counts[prefix]:
                self.probs[prefix][word] = counts[prefix][word] / total
            
    def get_possible_next_tokens(self, prefix):
        """
        :param prefix: string with space-separated prefix tokens
        :returns: a dictionary {token : it's probability} for all tokens with positive probabilities
        """
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.n + 1):]
        prefix = [ UNK ] * (self.n - 1 - len(prefix)) + prefix
        return self.probs[tuple(prefix)]
    
    def get_next_token_prob(self, prefix, next_token):
        """
        :param prefix: string with space-separated prefix tokens
        :param next_token: the next token to predict probability for
        :returns: P(next_token|prefix) a single number, 0 <= P <= 1
        """
        return self.get_possible_next_tokens(prefix).get(next_token, 0)

Let's test it!

In [9]:
dummy_lm = NGramLanguageModel(dummy_lines, n=3)

p_initial = dummy_lm.get_possible_next_tokens('') # '' -> ['_UNK_', '_UNK_']
assert np.allclose(p_initial['learning'], 0.02)
assert np.allclose(p_initial['a'], 0.13)
assert np.allclose(p_initial.get('meow', 0), 0)
assert np.allclose(sum(p_initial.values()), 1)

p_a = dummy_lm.get_possible_next_tokens('a') # '' -> ['_UNK_', 'a']
assert np.allclose(p_a['machine'], 0.15384615)
assert np.allclose(p_a['note'], 0.23076923)
assert np.allclose(p_a.get('the', 0), 0)
assert np.allclose(sum(p_a.values()), 1)

assert np.allclose(dummy_lm.get_possible_next_tokens('a note')['on'], 1)
assert dummy_lm.get_possible_next_tokens('a machine') == \
    dummy_lm.get_possible_next_tokens("there have always been ghosts in a machine"), \
    "your 3-gram model should only depend on 2 previous words"

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2086/2086 [00:00<00:00, 1129235.69it/s]


Now that you've got a working n-gram language model, let's see what sequences it can generate. But first, let's train it on the whole dataset.

In [10]:
lm = NGramLanguageModel(lines, n=3)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1219478/1219478 [00:03<00:00, 345212.06it/s]


The process of generating sequences is... well, it's sequential. You maintain a list of tokens and iteratively add next token by sampling with probabilities.

$ X = [] $

__forever:__
* $w_{next} \sim P(w_{next} | X)$
* $X = concat(X, w_{next})$


Instead of sampling with probabilities, one can also try always taking most likely token, sampling among top-K most likely tokens or sampling with temperature. In the latter case (temperature), one samples from

$$w_{next} \sim {P(w_{next} | X) ^ {1 / \tau} \over \sum_{\hat w} P(\hat w | X) ^ {1 / \tau}}$$

Where $\tau > 0$ is model temperature. If $\tau << 1$, more likely tokens will be sampled with even higher probability while less likely tokens will vanish.

In [11]:
def get_next_token(lm, prefix, temperature=1.0):
    """
    return next token after prefix;
    :param temperature: samples proportionally to lm probabilities ^ (1 / temperature)
        if temperature == 0, always takes most likely token. Break ties arbitrarily.
    """
    tokens_probs = lm.get_possible_next_tokens(prefix)
    tokens, probs = zip(*tokens_probs.items())
    if temperature == 0.0:
        return tokens[np.argmax(probs)]
    new_probs = np.power(probs, 1.0 / temperature)
    return np.random.choice(tokens, p=new_probs / new_probs.sum())

In [12]:
from collections import Counter
test_freqs = Counter([get_next_token(lm, 'there have') for _ in range(10000)])
assert 250 < test_freqs['not'] < 450
assert 8500 < test_freqs['been'] < 9500
assert 1 < test_freqs['lately'] < 200

test_freqs = Counter([get_next_token(lm, 'deep', temperature=1.0) for _ in range(10000)])
assert 1500 < test_freqs['learning'] < 3000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.5) for _ in range(10000)])
assert 8000 < test_freqs['learning'] < 9000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.0) for _ in range(10000)])
assert test_freqs['learning'] == 10000

print("Looks nice!")

Looks nice!


Let's have fun with this model

In [13]:
prefix = 'artificial' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break
        
print(prefix)

artificial neural network ( cnn ). the training set was very popular to encode complex features . in bioinformatics . inspired by previous psychophysical findings across the test accuracy in context of argumentation systems . however , an $ l_0 $-$ l_2 $ minimization , ( ii ) to predict the absorbance profile of the learning process . this project is expected to produce convergence controlled mutation and crossover operator is computed efficiently in closed form densities and policy makers or researchers . the evaluation of a compact and straightforward methods for linking entities in text generation ; semantic segmentation ,


In [14]:
prefix = 'bridging the' # <- more of your ideas

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.5)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break
        
print(prefix)

bridging the gap between the variables . we also compare the results of the most important parts of the proposed algorithm is evaluated on two datasets , and ( iii ) we show that the proposed method can be trained on the basic idea of this paper , we propose a method for the first time , we use a tracked object . in this paper , we propose a novel approach to the best of our proposed method can be viewed as a result , the proposed method is illustrated by using the popular theory about the future . _EOS_


__More in the homework:__ nucleus sampling, top-k sampling, beam search(not for the faint of heart).

### Evaluating language models: perplexity (1point)

Perplexity is a measure of how well your model approximates the true probability distribution behind the data. __Smaller perplexity = better model__.

To compute perplexity on one sentence, use:
$$
    {\mathbb{P}}(w_1 \dots w_N) = P(w_1, \dots, w_N)^{-\frac1N} = \left( \prod_t P(w_t \mid w_{t - n}, \dots, w_{t - 1})\right)^{-\frac1N},
$$


On the corpora level, perplexity is a product of probabilities of all tokens in all sentences to the power of $1/N$, where $N$ is __total length (in tokens) of all sentences__ in corpora.

This number can quickly get too small for float32/float64 precision, so we recommend you to first compute log-perplexity (from log-probabilities) and then take the exponent.

In [15]:
def perplexity(lm, lines, min_logprob=np.log(10 ** -50.)):
    """
    :param lines: a list of strings with space-separated tokens
    :param min_logprob: if log(P(w | ...)) is smaller than min_logprop, set it equal to min_logrob
    :returns: corpora-level perplexity - a single scalar number from the formula above
    
    Note: do not forget to compute P(w_first | empty) and P(eos | full_sequence)
    
    PLEASE USE lm.get_next_token_prob and NOT lm.get_possible_next_tokens
    """
    log_probs_sum = 0
    N = 0
    for line in tqdm(lines):
        tokens = line.split() + [EOS]
        N += len(tokens)
        for i in range(len(tokens)):
            prob = lm.get_next_token_prob(' '.join(tokens[:i]), tokens[i])
            if prob == 0:
                log_prob = min_logprob
            else:
                log_prob = max(min_logprob, np.log(prob))
            log_probs_sum += log_prob
    return np.exp(-1.0 / N * log_probs_sum)

In [16]:
lm1 = NGramLanguageModel(dummy_lines, n=1)
lm3 = NGramLanguageModel(dummy_lines, n=3)
lm10 = NGramLanguageModel(dummy_lines, n=10)

ppx1 = perplexity(lm1, dummy_lines)
ppx3 = perplexity(lm3, dummy_lines)
ppx10 = perplexity(lm10, dummy_lines)
ppx_missing = perplexity(lm3, ['the jabberwock , with eyes of flame , '])  # thanks, L. Carrol

print("Perplexities: ppx1=%.3f ppx3=%.3f ppx10=%.3f" % (ppx1, ppx3, ppx10))

assert all(0 < ppx < 500 for ppx in (ppx1, ppx3, ppx10)), "perplexity should be non-negative and reasonably small"
assert ppx1 > ppx3 > ppx10, "higher N models should overfit and "
assert np.isfinite(ppx_missing) and ppx_missing > 10 ** 6, "missing words should have large but finite perplexity. " \
    " Make sure you use min_logprob right"
assert np.allclose([ppx1, ppx3, ppx10], (318.2132342216302, 1.5199996213739575, 1.1838145037901249))

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 10672.53it/s]

Perplexities: ppx1=318.213 ppx3=1.520 ppx10=1.184


Now let's measure the actual perplexity: we'll split the data into train and test and score model on test data only.

In [17]:
from sklearn.model_selection import train_test_split
train_lines, test_lines = train_test_split(lines, test_size=0.25, random_state=42)

for n in (1, 2, 3):
    lm = NGramLanguageModel(n=n, lines=train_lines)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:07<00:00, 1314.19it/s]


N = 1, Perplexity = 1832.23136


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:08<00:00, 1190.77it/s]


N = 2, Perplexity = 85653987.28774


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:09<00:00, 1097.92it/s]

N = 3, Perplexity = 61999196259042902147072.00000


In [18]:
# whoops, it just blew up :)

### LM Smoothing

The problem with our simple language model is that whenever it encounters an n-gram it has never seen before, it assigns it with the probabilitiy of 0. Every time this happens, perplexity explodes.

To battle this issue, there's a technique called __smoothing__. The core idea is to modify counts in a way that prevents probabilities from getting too low. The simplest algorithm here is Additive smoothing (aka [Lapace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing)):

$$ P(w_t | prefix) = { Count(prefix, w_t) + \delta \over \sum_{\hat w} (Count(prefix, \hat w) + \delta) } $$

If counts for a given prefix are low, additive smoothing will adjust probabilities to a more uniform distribution. Not that the summation in the denominator goes over _all words in the vocabulary_.

Here's an example code we've implemented for you:

In [19]:
class LaplaceLanguageModel(NGramLanguageModel): 
    """ this code is an example, no need to change anything """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        counts = count_ngrams(lines, self.n)
        self.vocab = set(token for token_counts in counts.values() for token in token_counts)
        self.probs = defaultdict(Counter)

        for prefix in counts:
            token_counts = counts[prefix]
            total_count = sum(token_counts.values()) + delta * len(self.vocab)
            self.probs[prefix] = {token: (token_counts[token] + delta) / total_count
                                          for token in token_counts}
    def get_possible_next_tokens(self, prefix):
        token_probs = super().get_possible_next_tokens(prefix)
        missing_prob_total = 1.0 - sum(token_probs.values())
        missing_prob = missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        return {token: token_probs.get(token, missing_prob) for token in self.vocab}
    
    def get_next_token_prob(self, prefix, next_token):
        token_probs = super().get_possible_next_tokens(prefix)
        if next_token in token_probs:
            return token_probs[next_token]
        else:
            missing_prob_total = 1.0 - sum(token_probs.values())
            missing_prob_total = max(0, missing_prob_total) # prevent rounding errors
            return missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        

**Disclaimer**: the implementation above assumes all words unknown within a given context to be equally likely, *as well as the words outside of vocabulary*. Therefore, its' perplexity will be lower than it should when encountering such words. Therefore, comparing it with a model with fewer unknown words will not be fair. When implementing your own smoothing, you may handle this by adding a virtual `UNK` token of non-zero probability. Technically, this will result in a model where probabilities do not add up to $1$, but it is close enough for a practice excercise.

In [20]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = LaplaceLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 20392.38it/s]


In [21]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=0.1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:12<00:00, 843.28it/s]


N = 1, Perplexity = 1832.66878


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:11<00:00, 917.05it/s]


N = 2, Perplexity = 470.48021


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:11<00:00, 894.87it/s]

N = 3, Perplexity = 3679.44765


In [22]:
# optional: try to sample tokens from such a model

In [23]:
prefix = 'congratulations' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

congratulations boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boundedly boun

Не ну это просто AMAZING. Для `n=3` совсем все плохо.

In [24]:
lm = LaplaceLanguageModel(train_lines, n=2, delta=0.1)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30750/30750 [00:07<00:00, 4124.88it/s]


In [25]:
prefix = 'congratulations' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.1)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

congratulations behmann ' s performance of the proposed method for the proposed method is a new method for the proposed method for the proposed method for the proposed method for the proposed method for the proposed method is a new approach to the proposed method for the proposed method for the - based on the proposed method is a novel approach to the proposed method for the proposed method for the proposed method for the proposed method for the - based on the - based on the proposed method is a new method is a new approach to the proposed method


Для `n=2` perplexity меньше в 4 раза, но все равно плохое.

### Kneser-Ney smoothing (2 points)

Additive smoothing is simple, reasonably good but definitely not a State of The Art algorithm.


Your final task in this notebook is to implement [Kneser-Ney](https://en.wikipedia.org/wiki/Kneser%E2%80%93Ney_smoothing) smoothing.

It can be computed recurrently, for n>1:

$$P_{kn}(w_t | prefix_{n-1}) = { \max(0, Count(prefix_{n-1}, w_t) - \delta) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)} + \lambda_{prefix_{n-1}} \cdot P_{kn}(w_t | prefix_{n-2})$$

where
- $prefix_{n-1}$ is a tuple of {n-1} previous tokens
- $lambda_{prefix_{n-1}}$ is a normalization constant chosen so that probabilities add up to 1
- Unigram $P_{kn}(w_t | prefix_{n-2})$ corresponds to Kneser Ney smoothing for {N-1}-gram language model.
- Unigram $P_{kn}(w_t)$ is a special case: how likely it is to see x_t in an unfamiliar context

See lecture slides or wiki for more detailed formulae.

__Your task__ is to
- implement `KneserNeyLanguageModel` class,
- test it on 1-3 gram language models
- find optimal (within reason) smoothing delta for 3-gram language model with Kneser-Ney smoothing

In [26]:
class KneserNeyLanguageModel(NGramLanguageModel): 
    """ A template for Kneser-Ney language model. Default delta may be suboptimal. """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        self.delta = delta
        self.vocab = None

        self.cache      = defaultdict(lambda: {})
        self.base_probs = defaultdict(lambda: 0)  # probabilities for unigrams
        self.counts     = defaultdict(lambda: {})

        appeared        = defaultdict(lambda: {})
        for line in tqdm(lines, desc="Counting n-grams..."):
            line_tokens = line.split() + [ EOS ]
            for k in range(1, n + 1):
                tokens = [ UNK ] * (k - 1) + line_tokens
                for i in range(k - 1, len(tokens)):
                    if k == 1 and i < len(tokens) - 1:
                        prefix, word = (tokens[i],), tokens[i + 1]
                        if not appeared[(prefix, word)]:
                            appeared[(prefix, word)] = 1
                            self.base_probs[word] += 1
                    elif k > 1:
                        prefix, word = tuple(tokens[i - (k - 1):i]), tokens[i]
                        assert len(prefix) == k - 1
                        self.counts[prefix][word] = self.counts[prefix].get(word, 0) + 1
        for word in self.base_probs:
            self.base_probs[word] /= len(appeared)

        assert np.isclose(sum(self.base_probs.values()), 1.0, atol=1e-12)
        self.vocab = set(self.base_probs.keys())

    def _preprocess_prefix(self, prefix):
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.n + 1):]
        prefix = [ UNK ] * (self.n - 1 - len(prefix)) + prefix
        return tuple(prefix)

    def _get_word_prob(self, prefix, word):
        if self.cache[prefix].get(word, None) is None:
            p_kn = self._get_word_prob(prefix[1:], word) if len(prefix[1:]) > 0 else self.base_probs[word]

            prefix_counts = sum(self.counts[prefix].values())
            if prefix_counts == 0:
                prob = p_kn
            else:
                count = self.counts[prefix].get(word, 0)
                term1 = max(count - self.delta, 0)
                term2 = len(self.counts[prefix])
                prob = (term1 + self.delta * term2 * p_kn) / prefix_counts

            self.cache[prefix][word] = prob
        else:
            prob = self.cache[prefix][word]
        return prob

    def get_possible_next_tokens(self, prefix):
        # raise NotImplementerError("Lazy recursion!")
        prefix = self._preprocess_prefix(prefix)
        return {token: self._get_word_prob(prefix, token) for token in self.vocab}

    def get_next_token_prob(self, prefix, next_token):
        if self.n == 1:
            return self.base_probs[next_token]
        prefix = self._preprocess_prefix(prefix)
        assert len(prefix) == self.n - 1
        return self._get_word_prob(prefix, next_token)

In [27]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = KneserNeyLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

Counting n-grams...: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 28653.53it/s]


In [28]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:03<00:00, 2966.68it/s]


N = 1, Perplexity = 2882.02405


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:12<00:00, 790.26it/s]


N = 2, Perplexity = 405.14305


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:17<00:00, 584.90it/s]

N = 3, Perplexity = 312.14640


In [29]:
cnt = 0
for prefix in lm.counts:
    if len(prefix) == 2:
        cnt += 1
        if cnt < 1000:
            continue
        print(f"{prefix=}\n")
        print(f"{lm.counts[prefix]=}\n")
        print(f"{lm.cache[prefix]=}\n")
        break

prefix=('criteria', 'are')

lm.counts[prefix]={'developed': 1, 'different': 1, 'incomparable': 1, 'attained': 1, 'based': 2, 'represented': 1, 'typically': 1, 'asymptotically': 1, 'proposed': 1, 'not': 1, 'biased': 1, 'proven': 1, 'obtained': 2, 'decided': 1, 'suggested': 1, 'kept': 1, 'still': 1, 'analyzed': 1, 'used': 1, 'presented': 1, 'observational': 1, 'introduced': 1}

lm.cache[prefix]={'required': 0.003916308752801321, 'used': 0.02649680865945524, 'not': 0.02908254332274747, 'combined': 0.002863896270670463, 'proposed': 0.005692220825803687, 'able': 0.015325211511921148, 'satisfied': 0.0005842832753038249, 'useful': 0.0036828010777775386, 'derived': 0.0035972492099757385, 'the': 0.016162884125659766, 'widely': 0.0041086754283905175}



In [30]:
for delta in (0.1, 0.25, 0.5, 0.75, 1.0):
    lm = KneserNeyLanguageModel(train_lines, n=3, delta=delta)
    ppx = perplexity(lm, test_lines)
    print(f"DELTA = {delta}, N = {n}, Perplexity = {ppx:.5f}")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:17<00:00, 598.27it/s]


DELTA = 0.1, N = 3, Perplexity = 616.34782


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:17<00:00, 588.82it/s]


DELTA = 0.25, N = 3, Perplexity = 424.28287


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:17<00:00, 592.26it/s]


DELTA = 0.5, N = 3, Perplexity = 326.90305


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:17<00:00, 591.97it/s]


DELTA = 0.75, N = 3, Perplexity = 290.40452


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10250/10250 [00:17<00:00, 596.16it/s]

DELTA = 1.0, N = 3, Perplexity = 312.14640


Получили минимум для $\delta = 0.75$. Я погуглил в интернете, вроде как раз оптимальная $\delta \in [0.5, 0.75]$ ожидается для маленьких датасетов.

In [31]:
lm = KneserNeyLanguageModel(train_lines, n=3, delta=0.75)

Counting n-grams...: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30750/30750 [00:10<00:00, 2874.52it/s]


In [32]:
prefix = 'congratulations' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.1)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

congratulations , we propose a novel approach to the best of our approach is based on the other hand , the proposed method is based on the other hand , we propose a novel approach to the best of our approach is to use the same time , and the other hand , we propose a novel approach to the best of our approach is based on the other hand , we propose a novel approach to the best of our approach is based on the other hand , the proposed method is based on the other hand , the proposed


Что-то более осмысленное, хоть и зацикленное.